# Step 1: Environment setup

In [58]:
import os
import copy
import random
import collections
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

# Scikit-learn Preprocessing, Splitting, and Metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

# Imbalanced Learning
from imblearn.combine import SMOTETomek

# Classifiers
from xgboost import XGBClassifier
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Explainable AI (XAI)
import shap
import lime
import lime.lime_tabular

import preprocess

In [59]:
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Hardware calculation device: {device.type.upper()}")

[*] Hardware calculation device: CPU


In [60]:
filepath = "data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv"
df = preprocess.load_and_clean_dataset(filepath)

# Standardize binary target
df['Label'] = df['Label'].astype(str).str.strip().str.upper()
df['Label'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

X = df.drop(columns=['Label'])
y = df['Label'].values

# Split A: 70% Train, 30% Temp (Stratified)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# Split B: 15% Validation, 15% Test (Stratified)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

# Fit scaler strictly on Train split
scaler = MinMaxScaler()
scaler.fit(X_train)

feature_names = X_train.columns.tolist()
num_features = len(feature_names)

X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=feature_names)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=feature_names)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_names)

# Resample strictly on training fold
smote_tomek = SMOTETomek(random_state=42)
X_train_resampled, y_train_resampled = smote_tomek.fit_resample(
    X_train_scaled, y_train
)

print(f"[*] Preprocessing complete. Train: {X_train_resampled.shape}, Val: {X_val_scaled.shape}, Test: {X_test_scaled.shape}")

[*] Loading raw dataset from: data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
[*] Dataset Ingestion & Cleaning Audit:
    -> Raw Rows Ingested:          458,968
    -> Empty CSV Padding Purged:   288,602
    -> Valid Network Flows:        170,366
    -> Invalid Flows Dropped (Inf/NaN): 135
    -> Final Usable Flows:         170,231
[*] Preprocessing complete. Train: (235254, 78), Val: (25535, 78), Test: (25535, 78)


In [61]:
# Train & Calibrate Baseline XGBoost
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=0.1,
    random_state=42,
    use_label_encoder=False,
    early_stopping_rounds=15
)

xgb_model.fit(
    X_train_resampled, 
    y_train_resampled,
    eval_set=[(X_val_scaled, y_val)],
    verbose=False
)

c:\Users\Moritz\Documents\Uni\bachlor_thesis\ids-xai-thesis\thesis_env\Lib\site-packages\xgboost\callback.py:385: UserWarning: [11:08:46] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",15
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [62]:
# Train & Calibrate Baseline PyTorch DNN
class RobustNetworkSecurityDNN(nn.Module):
    def __init__(self, input_dim):
        super(RobustNetworkSecurityDNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(p=0.30)
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(p=0.30)
        self.fc3 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        x = self.dropout1(self.relu1(self.fc1(x)))
        x = self.dropout2(self.relu2(self.fc2(x)))
        return self.sigmoid(self.fc3(x))

model_dnn = RobustNetworkSecurityDNN(input_dim=num_features).to(device)

train_dataset = TensorDataset(
    torch.FloatTensor(X_train_resampled.values),
    torch.FloatTensor(y_train_resampled).unsqueeze(1)
)
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
val_x_tensor = torch.FloatTensor(X_val_scaled.values).to(device)
val_y_tensor = torch.FloatTensor(y_val).unsqueeze(1).to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model_dnn.parameters(), lr=0.001)

patience = 10
best_val_loss = float('inf')
best_model_weights = None
patience_counter = 0

for epoch in range(1, 151):
    model_dnn.train()
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        loss = criterion(model_dnn(batch_x), batch_y)
        loss.backward()
        optimizer.step()
        
    model_dnn.eval()
    with torch.no_grad():
        val_loss = criterion(model_dnn(val_x_tensor), val_y_tensor).item()
        
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_weights = copy.deepcopy(model_dnn.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        
    if patience_counter >= patience:
        break

if best_model_weights is not None:
    model_dnn.load_state_dict(best_model_weights)

In [63]:
# Inference Wrappers and Decision Calibration
def get_dnn_probabilities(df_input):
    model_dnn.eval()
    with torch.no_grad():
        t_in = torch.FloatTensor(df_input.values).to(device)
        probs = model_dnn(t_in).cpu().numpy().flatten()
    return probs

def xgb_predict_proba_wrapper(x_numpy):
    df_temp = pd.DataFrame(x_numpy, columns=feature_names)
    return xgb_model.predict_proba(df_temp)

def dnn_predict_proba_wrapper(x_numpy):
    df_temp = pd.DataFrame(x_numpy, columns=feature_names)
    probs_class_1 = get_dnn_probabilities(df_temp)
    probs_class_0 = 1.0 - probs_class_1
    return np.column_stack((probs_class_0, probs_class_1))

# Validation Threshold Search
xgb_val_probs = xgb_model.predict_proba(X_val_scaled)[:, 1]
dnn_val_probs = get_dnn_probabilities(X_val_scaled)

thresholds_pool = np.arange(0.50, 0.96, 0.05)
best_xgb_threshold, best_xgb_f1 = 0.50, 0.0
best_dnn_threshold, best_dnn_f1 = 0.50, 0.0

for t in thresholds_pool:
    f1_x = f1_score(y_val, (xgb_val_probs >= t).astype(int), zero_division=0)
    if f1_x > best_xgb_f1:
        best_xgb_f1, best_xgb_threshold = f1_x, t
        
    f1_d = f1_score(y_val, (dnn_val_probs >= t).astype(int), zero_division=0)
    if f1_d > best_dnn_f1:
        best_dnn_f1, best_dnn_threshold = f1_d, t

print(f"[*] Calibrated Thresholds: XGBoost = {best_xgb_threshold:.2f} | DNN = {best_dnn_threshold:.2f}")

# Extract True Positive Attack Cohorts from Test Set
xgb_test_probs = xgb_model.predict_proba(X_test_scaled)[:, 1]
dnn_test_probs = get_dnn_probabilities(X_test_scaled)

xgb_tp_indices = np.where((y_test == 1) & (xgb_test_probs >= best_xgb_threshold))[0]
dnn_tp_indices = np.where((y_test == 1) & (dnn_test_probs >= best_dnn_threshold))[0]

print(f"[*] Identified Eligible True Positive Attacks in Test Split:")
print(f"    -> XGBoost TP count: {len(xgb_tp_indices)}")
print(f"    -> DNN TP count:     {len(dnn_tp_indices)}")

[*] Calibrated Thresholds: XGBoost = 0.95 | DNN = 0.80
[*] Identified Eligible True Positive Attacks in Test Split:
    -> XGBoost TP count: 323
    -> DNN TP count:     308


In [64]:

# Exact Unchanged Baseline Explainers
# 1. Primary Continuous LIME (KW=2.5)
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_scaled.values,
    feature_names=feature_names,
    class_names=["Benign", "Web Attack"],
    mode="classification",
    discretize_continuous=False,
    kernel_width=2.5,
    random_state=42
)

# 2. Exact Baseline SHAP Explainers
shap_background_baseline = shap.sample(X_train_scaled, 100, random_state=42)

shap_explainer_xgb_tree = shap.TreeExplainer(model=xgb_model)
shap_explainer_dnn = shap.KernelExplainer(
    model=dnn_predict_proba_wrapper,
    data=shap_background_baseline
)

def extract_lime_vector(instance, predict_fn):
    """Extracts Class 1 LIME attributions aligned by feature index."""
    exp = lime_explainer.explain_instance(
        data_row=instance,
        predict_fn=predict_fn,
        labels=(1,),
        num_features=num_features
    )
    vec = np.zeros(num_features)
    for f_idx, weight in exp.local_exp[1]:
        vec[f_idx] = weight
    return vec, exp.score

def extract_shap_vector(instance, explainer):
    """Extracts Class 1 SHAP attributions matching baseline setup, suppressing tqdm bars."""
    if isinstance(explainer, shap.TreeExplainer):
        df_in = pd.DataFrame([instance], columns=feature_names)
        raw_vals = explainer.shap_values(df_in)
        if isinstance(raw_vals, list):
            return raw_vals[1].flatten()
        elif isinstance(raw_vals, np.ndarray) and raw_vals.ndim == 3:
            return raw_vals[0, :, 1]
        return np.array(raw_vals).flatten()
    else:
        raw_vals = explainer.shap_values(instance.reshape(1, -1), nsamples=100, silent=True)
        if isinstance(raw_vals, list):
            return raw_vals[1].flatten()
        elif isinstance(raw_vals, np.ndarray) and raw_vals.ndim == 3:
            return raw_vals[0, :, 1]
        return np.array(raw_vals).flatten()

print("[+] Baseline explainers initialized with verified baseline configurations.")

[+] Baseline explainers initialized with verified baseline configurations.


# Step 2: Feasibility Constraints & Targeted I-Attack Formulation

In [65]:

# 1. Locate target feature Init_Win_bytes_forward
target_candidates = [c for c in feature_names if "init_win_bytes_forward" in c.lower().replace(" ", "_")]
TARGET_FEATURE = target_candidates[0] if target_candidates else feature_names[0]
TARGET_IDX = feature_names.index(TARGET_FEATURE)

# 2. Benign reference value (median in scaled training set for y=0)
benign_reference_val = float(X_train_scaled.loc[y_train == 0, TARGET_FEATURE].median())

# 3. Select first True Positive attack sample from test set for validation
sample_tp_idx = xgb_tp_indices[0]
sample_row = X_test_scaled.iloc[sample_tp_idx].values

print(f"[*] Targeted Feature for I-Attack:          '{TARGET_FEATURE}' (Index: {TARGET_IDX})")
print(f"[*] Sample's current value of {TARGET_FEATURE}: {sample_row[TARGET_IDX]:.6f}")
print(f"[*] Benign reference value:                  {benign_reference_val:.6f}")
print(f"[*] Distance to benign reference:           {abs(sample_row[TARGET_IDX] - benign_reference_val):.6f}")

[*] Targeted Feature for I-Attack:          'Init_Win_bytes_forward' (Index: 66)
[*] Sample's current value of Init_Win_bytes_forward: 0.445572
[*] Benign reference value:                  0.000351
[*] Distance to benign reference:           0.445221


In [66]:

def execute_targeted_i_attack(
    x_orig,
    predict_fn,
    model_threshold,
    explainer_func,
    epsilon=0.02,
    delta_p=0.10,
    search_steps=10
):
    """
    Executes a bounded, prediction-preserving targeted explanation attack (I-Attack).
    """
    orig_p1 = predict_fn(x_orig.reshape(1, -1))[0, 1]
    orig_attr = explainer_func(x_orig)
    
    orig_rank_order = np.argsort(np.abs(orig_attr))[::-1]
    orig_target_rank = int(np.where(orig_rank_order == TARGET_IDX)[0][0]) + 1
    orig_target_attr = orig_attr[TARGET_IDX]
    
    direction = -1.0 if x_orig[TARGET_IDX] > benign_reference_val else 1.0
    
    best_x_adv = x_orig.copy()
    best_attr = orig_attr.copy()
    best_rank = orig_target_rank
    attack_success = False
    pred_preserved = True
    
    # Candidate steps from fine to full epsilon budget
    step_sizes = np.linspace(0.1 * epsilon, epsilon, search_steps)
    
    for step in step_sizes:
        x_candidate = x_orig.copy()
        x_candidate[TARGET_IDX] = np.clip(x_candidate[TARGET_IDX] + direction * step, 0.0, 1.0)
        
        cand_p1 = predict_fn(x_candidate.reshape(1, -1))[0, 1]
        
        # Check prediction-preservation constraints
        is_still_attack = cand_p1 >= model_threshold
        is_within_drift = abs(orig_p1 - cand_p1) <= delta_p
        
        if is_still_attack and is_within_drift:
            cand_attr = explainer_func(x_candidate)
            cand_rank_order = np.argsort(np.abs(cand_attr))[::-1]
            cand_target_rank = int(np.where(cand_rank_order == TARGET_IDX)[0][0]) + 1
            
            # Demotion criteria: rank drops by >= 3 OR target drops out of Top-5
            is_demoted = (cand_target_rank - orig_target_rank >= 3) or (orig_target_rank <= 5 and cand_target_rank > 5)
            
            if is_demoted:
                best_x_adv = x_candidate
                best_attr = cand_attr
                best_rank = cand_target_rank
                attack_success = True
                break
            elif cand_target_rank > best_rank:
                best_x_adv = x_candidate
                best_attr = cand_attr
                best_rank = cand_target_rank
        else:
            # Reached boundary
            break
            
    adv_p1 = predict_fn(best_x_adv.reshape(1, -1))[0, 1]
    perturbation = best_x_adv - x_orig
    
    return {
        "orig_p1": orig_p1,
        "adv_p1": adv_p1,
        "delta_p": abs(orig_p1 - adv_p1),
        "orig_rank": orig_target_rank,
        "adv_rank": best_rank,
        "rank_drop": best_rank - orig_target_rank,
        "orig_target_attr": orig_target_attr,
        "adv_target_attr": best_attr[TARGET_IDX],
        "attr_delta": abs(orig_target_attr - best_attr[TARGET_IDX]),
        "pred_preserved": int((adv_p1 >= model_threshold) and (abs(orig_p1 - adv_p1) <= delta_p)),
        "attack_success": int(attack_success),
        "orig_attr": orig_attr,
        "adv_attr": best_attr,
        "x_adv": best_x_adv,
        "L0": int(np.sum(perturbation != 0)),
        "Linf": float(np.max(np.abs(perturbation)))
    }

In [67]:

# Execute verification sanity check using XGBoost + TreeSHAP
sanity_check_res = execute_targeted_i_attack(
    x_orig=sample_row,
    predict_fn=xgb_predict_proba_wrapper,
    model_threshold=best_xgb_threshold,
    explainer_func=lambda x: extract_shap_vector(x, shap_explainer_xgb_tree),
    epsilon=0.02,
    delta_p=0.10
)

print("=================== UPDATED SANITY CHECK ===================")
print(f"Original Prediction P(Attack):      {sanity_check_res['orig_p1']:.4f}")
print(f"Adversarial Prediction P(Attack):   {sanity_check_res['adv_p1']:.4f}")
print(f"Prediction Preserved (|Δp| <= 0.1): {sanity_check_res['pred_preserved']}")
print(f"Target Feature Value Shift:         {sample_row[TARGET_IDX]:.4f} -> {sanity_check_res['x_adv'][TARGET_IDX]:.4f}")
print(f"Target Feature Attribution:         {sanity_check_res['orig_target_attr']:.4f} -> {sanity_check_res['adv_target_attr']:.4f} (Δ = {sanity_check_res['attr_delta']:.4f})")
print(f"Target Feature Rank Shift:          Rank {sanity_check_res['orig_rank']} -> Rank {sanity_check_res['adv_rank']} (Δ = {sanity_check_res['rank_drop']})")
print(f"I-Attack Success:                   {sanity_check_res['attack_success']}")
print(f"Perturbation Norms:                 L0 = {sanity_check_res['L0']}, Linf = {sanity_check_res['Linf']:.4f}")
print("============================================================")

=================== UPDATED SANITY CHECK ===================
Original Prediction P(Attack):      1.0000
Adversarial Prediction P(Attack):   1.0000
Prediction Preserved (|Δp| <= 0.1): 1
Target Feature Value Shift:         0.4456 -> 0.4456
Target Feature Attribution:         1.7667 -> 1.7667 (Δ = 0.0000)
Target Feature Rank Shift:          Rank 2 -> Rank 2 (Δ = 0)
I-Attack Success:                   0
Perturbation Norms:                 L0 = 0, Linf = 0.0000


# Step 3: Systematic Multi-Observation Evaluation Loop

In [68]:
# Select Evaluation Cohort (True Positives from untouched test split)
# Deterministically sample up to 25 True Positives per model architecture
rng = np.random.default_rng(42)
N_EVAL = 25

eval_xgb_tp = rng.choice(xgb_tp_indices, size=min(N_EVAL, len(xgb_tp_indices)), replace=False)
eval_dnn_tp = rng.choice(dnn_tp_indices, size=min(N_EVAL, len(dnn_tp_indices)), replace=False)

print(f"[*] Evaluation cohort isolated:")
print(f"    -> XGBoost cohort size: {len(eval_xgb_tp)} alerts")
print(f"    -> DNN cohort size:     {len(eval_dnn_tp)} alerts")

# Tested perturbation budgets
budgets = [0.005, 0.01, 0.02, 0.05]

[*] Evaluation cohort isolated:
    -> XGBoost cohort size: 25 alerts
    -> DNN cohort size:     25 alerts


In [69]:
# Helper function to compute attribution similarity metrics
def compute_explanation_similarity(attr1, attr2, k=5):
    """
    Computes Top-k Jaccard similarity and Union Spearman rank correlation
    between original and adversarial attribution vectors.
    """
    top_k_orig = set(np.argsort(np.abs(attr1))[-k:])
    top_k_adv = set(np.argsort(np.abs(attr2))[-k:])
    
    intersection = top_k_orig.intersection(top_k_adv)
    union = top_k_orig.union(top_k_adv)
    
    jaccard = len(intersection) / len(union) if len(union) > 0 else 0.0
    
    union_indices = list(union)
    slice1 = attr1[union_indices]
    slice2 = attr2[union_indices]
    
    if np.std(slice1) == 0 or np.std(slice2) == 0 or len(union_indices) < 2:
        rho = 0.0
    else:
        rho, _ = stats.spearmanr(slice1, slice2)
        rho = 0.0 if np.isnan(rho) else rho
        
    return jaccard, rho

In [70]:
attack_evaluation_records = []
budgets = [0.005, 0.01, 0.02, 0.05]

pipeline_configs = [
    {
        'model_name': 'DNN',
        'explainer_name': 'KernelSHAP',
        'predict_fn': dnn_predict_proba_wrapper,
        'threshold': best_dnn_threshold,
        'explainer_fn': lambda x: extract_shap_vector(x, shap_explainer_dnn),
        'cohort': eval_dnn_tp
    },
    {
        'model_name': 'DNN',
        'explainer_name': 'LIME',
        'predict_fn': dnn_predict_proba_wrapper,
        'threshold': best_dnn_threshold,
        'explainer_fn': lambda x: extract_lime_vector(x, dnn_predict_proba_wrapper)[0],
        'cohort': eval_dnn_tp
    },
    {
        'model_name': 'XGBoost',
        'explainer_name': 'TreeSHAP',
        'predict_fn': xgb_predict_proba_wrapper,
        'threshold': best_xgb_threshold,
        'explainer_fn': lambda x: extract_shap_vector(x, shap_explainer_xgb_tree),
        'cohort': eval_xgb_tp
    },
    {
        'model_name': 'XGBoost',
        'explainer_name': 'LIME',
        'predict_fn': xgb_predict_proba_wrapper,
        'threshold': best_xgb_threshold,
        'explainer_fn': lambda x: extract_lime_vector(x, xgb_predict_proba_wrapper)[0],
        'cohort': eval_xgb_tp
    }
]

In [71]:
print(f"[*] Running Monotonic Cumulative I-Attack Evaluation across budgets...")

for pipe in pipeline_configs:
    m_name = pipe['model_name']
    e_name = pipe['explainer_name']
    p_fn = pipe['predict_fn']
    thresh = pipe['threshold']
    exp_fn = pipe['explainer_fn']
    cohort = pipe['cohort']
    
    print(f"\n---> Auditing: {m_name} + {e_name} (Cohort N = {len(cohort)})")
    
    # Store successful adversarial results from smaller budgets
    successful_adversaries = {}
    
    for eps in budgets:
        print(f"     Budget ε = {eps}...")
        for test_idx in cohort:
            x_inst = X_test_scaled.iloc[test_idx].values
            
            # If already broken at a smaller epsilon, carry forward to guarantee monotonicity
            if test_idx in successful_adversaries:
                res = copy.deepcopy(successful_adversaries[test_idx])
            else:
                res = execute_targeted_i_attack(
                    x_orig=x_inst,
                    predict_fn=p_fn,
                    model_threshold=thresh,
                    explainer_func=exp_fn,
                    epsilon=eps,
                    delta_p=0.10,
                    search_steps=10
                )
                if res['attack_success'] == 1:
                    successful_adversaries[test_idx] = res
                    
            jacc_5, rho_5 = compute_explanation_similarity(res['orig_attr'], res['adv_attr'], k=5)
            
            attack_evaluation_records.append({
                'Model': m_name,
                'Explainer': e_name,
                'Budget_Epsilon': eps,
                'Test_Idx': test_idx,
                'Pred_Preserved': res['pred_preserved'],
                'Attack_Success': res['attack_success'],
                'Orig_Rank': res['orig_rank'],
                'Adv_Rank': res['adv_rank'],
                'Rank_Drop': res['rank_drop'],
                'Delta_P': res['delta_p'],
                'Top5_Jaccard': jacc_5,
                'Top5_Spearman': rho_5,
                'L0': res['L0'],
                'Linf': res['Linf']
            })

df_attack_results = pd.DataFrame(attack_evaluation_records)
print("\n[+] Monotonic evaluation complete.")

[*] Running Monotonic Cumulative I-Attack Evaluation across budgets...

---> Auditing: DNN + KernelSHAP (Cohort N = 25)
     Budget ε = 0.005...
     Budget ε = 0.01...
     Budget ε = 0.02...
     Budget ε = 0.05...

---> Auditing: DNN + LIME (Cohort N = 25)
     Budget ε = 0.005...
     Budget ε = 0.01...
     Budget ε = 0.02...
     Budget ε = 0.05...

---> Auditing: XGBoost + TreeSHAP (Cohort N = 25)
     Budget ε = 0.005...
     Budget ε = 0.01...
     Budget ε = 0.02...
     Budget ε = 0.05...

---> Auditing: XGBoost + LIME (Cohort N = 25)
     Budget ε = 0.005...
     Budget ε = 0.01...
     Budget ε = 0.02...
     Budget ε = 0.05...

[+] Monotonic evaluation complete.


In [72]:
print(df_attack_results[['Model', 'Explainer']].value_counts())

Model    Explainer 
DNN      KernelSHAP    100
         LIME          100
XGBoost  TreeSHAP      100
         LIME          100
Name: count, dtype: int64


# Step 4: Vulnerability Analysis & Academic Curves

In [73]:
# Group results across Model, Explainer, and Perturbation Budget
summary_table = df_attack_results.groupby(['Model', 'Explainer', 'Budget_Epsilon']).agg(
    N_Samples=('Test_Idx', 'count'),
    Prediction_Preservation_Rate=('Pred_Preserved', lambda x: f"{np.mean(x)*100:.1f}%"),
    Attack_Success_Rate=('Attack_Success', lambda x: f"{np.mean(x)*100:.1f}%"),
    Mean_Rank_Drop=('Rank_Drop', 'mean'),
    Mean_Top5_Jaccard=('Top5_Jaccard', 'mean'),
    Mean_Top5_Spearman=('Top5_Spearman', 'mean'),
    Mean_Prob_Drift=('Delta_P', 'mean'),
    Mean_Linf=('Linf', 'mean')
).reset_index()

print("\n" + "="*120)
print("              SYSTEMATIC I-ATTACK VULNERABILITY SUMMARY")
print("="*120)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(summary_table.to_string(index=False))
print("="*120)


              SYSTEMATIC I-ATTACK VULNERABILITY SUMMARY
  Model  Explainer  Budget_Epsilon  N_Samples Prediction_Preservation_Rate Attack_Success_Rate  Mean_Rank_Drop  Mean_Top5_Jaccard  Mean_Top5_Spearman  Mean_Prob_Drift  Mean_Linf
    DNN KernelSHAP           0.005         25                       100.0%               64.0%            7.60           0.368254           -0.117304     1.420975e-05    0.00070
    DNN KernelSHAP           0.010         25                       100.0%               84.0%           10.28           0.164286           -0.464622     3.546238e-05    0.00130
    DNN KernelSHAP           0.020         25                       100.0%               92.0%           10.96           0.157143           -0.476554     3.899336e-05    0.00142
    DNN KernelSHAP           0.050         25                       100.0%               96.0%           11.20           0.157143           -0.476554     7.270098e-05    0.00186
    DNN       LIME           0.005         25        